In [1]:
import sys
sys.path.insert(0, '../iaml')

import pandas as pd
import json

from iaml import *

[nltk_data] Downloading package stopwords to
[nltk_data]     /home/0421485/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package punkt to /home/0421485/nltk_data...
[nltk_data]   Package punkt is already up-to-date!


[08:12:11] cuDF not found: falling back to standalone pandas.

In [2]:
# Load dataset used for the test
dataset = pd.read_csv('../perf_logger/tests_data/titanic.csv', delimiter=';')

y = dataset['label']
X = dataset.drop(columns=['label'])

dataset.head()

,label,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,0,3,Braund; Mr. Owen Harris,male,22.0,1,0,A/5 21171,7.2500,NaN,S
1,1,1,Cumings; Mrs. John Bradley (Florence Briggs Th...,female,38.0,1,0,PC 17599,71.2833,C85,C
2,1,3,Heikkinen; Miss. Laina,female,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S
3,1,1,Futrelle; Mrs. Jacques Heath (Lily May Peel),female,35.0,1,0,113803,53.1000,C123,S
4,0,3,Allen; Mr. William Henry,male,35.0,0,0,373450,8.0500,NaN,S


In [5]:
# Get default Pipeline & disable a step
default_pipeline = IAML().json_pipeline()
default_pipeline['children'][0]['enable'] = False # Desactivation de la couche de pretraitement avec le MinMaxScaler

# Dans l'exemple, l'étape désactivée est une metastep, cela désactive donc aussi ses enfants et continue naturellement.
# -> C'est le comportement souhaiter, ça permet de désactiver une grande étape de traitement comme "cleaning" par exemple. 
# Il est aussi possible de désactiver une étape plus classique, cette dernière ne sera donc pas utilisée pour la génération des candidats.

[08:12:37] Max duration of each stage was set to 900 seconds

[{'step': 'MetaStep',
  'name': 'MetaStep',
  'description': 'Step description...',
  'enable': False,
  'configuration': {},
  'children': [{'step': 'ActDateConverter',
    'name': 'Convert Short text to date if possible',
    'description': 'Step description...',
    'enable': True,
    'configuration': {'authorized_error_ratios': {'default': 0.05,
      'description': 'Over this ratios, the column will not be converted into date',
      'value': 0.05},
     'sample_size': {'default': 200,
      'description': 'Convert date is time consuming.                     To save time, date detection will be done on a random sample.                     Set to -1 to detect on the whole dataset',
      'value': 200}},
    'children': []}]},
 {'step': 'MetaStep',
  'name': 'MetaStep',
  'description': 'Step description...',
  'enable': True,
  'configuration': {},
  'children': [{'step': 'ActDropNumericalColumn',
    'name': 'Drop numerical columns',
    'description': 'Drop numerical columns whe

In [6]:
# Training with a disabled step
iaml = IAML(max_duration=120, quiet=True)
iaml.first_step = Step.from_pipeline(default_pipeline) # Modification du pipeline avec la step désactivée
candidates = iaml.fit(pd.DataFrame(X), y)

[08:12:41] 13 generated pipelines

[08:13:44] gen1,             Nb mutation=-1,             Nb random=27

[08:14:39] gen2,             Nb mutation=31,             Nb random=0

[08:14:40] gen3,             Nb mutation=31,             Nb random=0

In [7]:
iaml.chosen_model # Si la desactivation fonctionne, il ne dois pas y avoir de MinMaxScaler

IAMLPipeline(estimator_type='classifier',
             original_dataset=     Pclass                               Name     Sex   Age  SibSp  Parch  \
459       3              O'Connor; Mr. Maurice    male   NaN      0      0   
536       1  Butt; Major. Archibald Willingham    male  45.0      0      0   
252       1          Stead; Mr. William Thomas    male  62.0      0      0   
624       3        Bowen; Mr. David John "Dai"    male  21.0      0      0   
502       3     O'Sullivan; Miss. Bridget Mary  female   NaN      0      0   
..      ...                                ...     ...   ...    ...    ...   
411       3                    Hart; Mr. Henry    male   NaN      0      0   
295       1                  Lewy; Mr. Ervin G    male   NaN      0      0   
3...
                     <iaml.actionables.features_selection.act_remove_high_correlated_column.ActRemoveHighCorrelatedColumn object at 0x7effc6c65b40>),
                    ('Min Max Scaler',
                     <iaml.actionables.normalize.act_minmax_scaler.ActMinMaxScaler object at 0x7effc6c66b90>),
                    ('VoidStep',
                     <iaml.void_step.VoidStep object at 0x7eff2fdcd300>),
                    ('Learn : SVM Classification',
                     <iaml.actionables.predictors.act_svm_svc.ActSVMSVC object at 0x7eff2fdf44f0>)])